# Retail Vision: Real-Time Safety-Compliance Detection with YOLOv8
**Deep Learning course project: YOLO module**

**Business problem.** Sites (construction, warehouses, retail back-of-house, factories) must make sure workers wear helmets, vests, gloves, boots and goggles. Manual inspection is slow, inconsistent and does not scale. We train an object detector that finds PPE (and missing PPE) in images/video, then turn its raw detections into a **compliance alert system**.

**What this notebook does (end to end)**
1. Data audit (class balance, box sizes, spatial distribution, samples)
2. Controlled experiments: **4 optimizers** (SGD, Adam, AdamW, RMSProp) on YOLOv8n, then a **model-size comparison** (YOLOv8n vs YOLOv8s)
3. Honest evaluation: model chosen on *validation*, reported on the held-out *test* split (mAP, per-class AP, PR/F1 curves, confusion matrix)
4. Operating-point tuning (best confidence threshold from the F1 curve)
5. **Application layer:** image-level "violation" alerts, precision/recall trade-off, safety-first threshold
6. **Deployment optimisation:** FP16 on GPU, ONNX export on CPU, latency/FPS table
7. Annotated demo video for the presentation
8. One-click export of every figure/table for slides

**Runtime:** Colab → *Runtime → Change runtime type → T4 GPU*. Full run ≈ 45–60 min. Set `QUICK_TEST = True` first for a 5-minute smoke test.

> **Local CPU version** (no NVIDIA GPU). Reduced settings: 416 px, half the training data, 2 optimizers (SGD, AdamW), no YOLOv8s run. Results are indicative, not comparable to the full GPU run.

## 1. Setup and configuration

In [ ]:
# ======================= CONFIG (local CPU version) =======================
import os
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"   # set by the terminal: $env:QUICK_TEST="1"
EPOCHS     = 10          # per training run
IMGSZ      = 416         # smaller images = much faster on CPU
BATCH      = 8
SEED       = 42
USE_DRIVE  = False
FRACTION   = 0.4         # fraction of the training set used (CPU time saver)
RUN_V8S    = False       # YOLOv8s comparison is too slow on CPU; set True only if you have time
DATA_YAML  = "construction-ppe.yaml"
# ==========================================================================
if QUICK_TEST:
    EPOCHS, FRACTION = 1, 0.05

In [ ]:
import os, json, time, random, shutil, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, seaborn as sns
import torch
from IPython.display import display, Image as IPImage
warnings.filterwarnings("ignore")

import ultralytics
from ultralytics import YOLO
ultralytics.checks()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime to T4 GPU!)")

WORK = Path("./yolo_retail_cpu" + ("_quick" if QUICK_TEST else ""))
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = Path("/content/drive/MyDrive/yolo_retail" + ("_quick" if QUICK_TEST else ""))
    except Exception as e:
        print("Drive not mounted, using local disk:", e)
RUNS, FIGS = WORK / "runs", WORK / "figures"
for d in (RUNS, FIGS): d.mkdir(parents=True, exist_ok=True)
print("Outputs ->", WORK)

sns.set_theme(style="whitegrid", context="talk", palette="deep")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
def save(fig, name):
    fig.savefig(FIGS / f"{name}.png"); plt.show()

## 2. Data audit
Before training anything we check what the model will actually learn from: class balance, object sizes (small objects are what YOLO struggles with) and where objects sit in the frame.

In [ ]:
from ultralytics.data.utils import check_det_dataset
try:
    info = check_det_dataset(DATA_YAML)          # downloads on first call
except Exception as e:
    raise SystemExit(f"Could not load {DATA_YAML}: {e}\nTry: %pip install -U ultralytics, then restart the runtime.")

NAMES = {int(k): str(v) for k, v in info["names"].items()}
NC = len(NAMES)
first = lambda p: p[0] if isinstance(p, (list, tuple)) else p
SPLITS = {s: Path(first(info[s])) for s in ("train", "val", "test") if info.get(s)}
print("Classes:", NAMES)
print("Splits :", {k: str(v) for k, v in SPLITS.items()})

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def list_images(d): return sorted(p for p in Path(d).rglob("*") if p.suffix.lower() in IMG_EXT)
def lbl_path(img):
    parts = list(img.parts)
    i = max(k for k, p in enumerate(parts) if p == "images")
    parts[i] = "labels"
    return Path(*parts).with_suffix(".txt")

rows = []
for split, d in SPLITS.items():
    for im in list_images(d):
        lp, n = lbl_path(im), 0
        if lp.exists():
            for line in lp.read_text().strip().splitlines():
                v = line.split()
                if len(v) < 5: continue
                c, cx, cy, w, h = int(float(v[0])), *map(float, v[1:5])
                rows.append((split, im.name, str(im), c, cx, cy, w, h)); n += 1
        if n == 0:
            rows.append((split, im.name, str(im), -1, np.nan, np.nan, np.nan, np.nan))   # background image
ann = pd.DataFrame(rows, columns=["split", "image", "path", "cls", "cx", "cy", "w", "h"])
boxes = ann[ann.cls >= 0].copy()
boxes["cls_name"] = boxes.cls.map(NAMES)
boxes["area"] = boxes.w * boxes.h

summary = ann.groupby("split").agg(images=("image", "nunique")).join(boxes.groupby("split").size().rename("boxes"))
summary["boxes_per_image"] = (summary.boxes / summary.images).round(2)
display(summary)

In [ ]:
# --- class balance per split
fig, ax = plt.subplots(figsize=(11, 6))
cc = boxes.groupby(["cls_name", "split"]).size().unstack(fill_value=0)
cc = cc.loc[cc.sum(1).sort_values().index]
cc.plot.barh(ax=ax, width=0.8)
ax.set_title("Annotations per class"); ax.set_xlabel("# boxes"); ax.set_ylabel("")
save(fig, "01_class_balance")

# --- object size + objects per image + spatial heatmap
fig, axs = plt.subplots(1, 3, figsize=(20, 5.5))
sns.histplot(np.sqrt(boxes.area) * IMGSZ, bins=40, ax=axs[0], color="C0")
axs[0].axvline(32, color="r", ls="--"); axs[0].text(34, axs[0].get_ylim()[1] * 0.9, "COCO 'small'\n(<32 px)", color="r", fontsize=11)
axs[0].set_title(f"Object size (px @{IMGSZ})"); axs[0].set_xlabel("sqrt(box area)")
npi = ann.groupby("image").cls.apply(lambda s: (s >= 0).sum())
sns.histplot(npi, bins=range(0, int(npi.max()) + 2), ax=axs[1], color="C2")
axs[1].set_title("Objects per image"); axs[1].set_xlabel("# objects")
h2d = np.histogram2d(boxes.cx, boxes.cy, bins=30, range=[[0, 1], [0, 1]])[0].T
axs[2].imshow(h2d, origin="upper", cmap="magma"); axs[2].set_title("Where objects appear"); axs[2].axis("off")
plt.tight_layout(); save(fig, "02_box_stats")
small = (np.sqrt(boxes.area) * IMGSZ < 32).mean()
print(f"{small:.1%} of boxes are 'small' (<32px at {IMGSZ}px) -> expect lower AP on small items like gloves/goggles.")

In [ ]:
# --- ground-truth samples
COLORS = {i: tuple(int(x) for x in np.random.RandomState(i * 7 + 3).randint(40, 230, 3)) for i in NAMES}
def draw_gt(p):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); H, W = im.shape[:2]
    lp = lbl_path(Path(p))
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            v = line.split()
            if len(v) < 5: continue
            c = int(float(v[0])); cx, cy, bw, bh = map(float, v[1:5])
            x1, y1, x2, y2 = int((cx - bw/2) * W), int((cy - bh/2) * H), int((cx + bw/2) * W), int((cy + bh/2) * H)
            cv2.rectangle(im, (x1, y1), (x2, y2), COLORS[c], max(2, W // 350))
            cv2.putText(im, NAMES[c], (x1, max(18, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, max(0.5, W / 1500), COLORS[c], max(1, W // 600))
    return im

rng = random.Random(SEED)
pick = rng.sample(list_images(SPLITS["train"]), 8)
fig, axs = plt.subplots(2, 4, figsize=(22, 11))
for a, p in zip(axs.ravel(), pick): a.imshow(draw_gt(p)); a.axis("off")
plt.tight_layout(); save(fig, "03_gt_samples")

## 3. Controlled experiments
**Design (so the comparison is fair):** same data, same seed, same image size, same epochs, same augmentation. Only one factor changes at a time.

* **Experiment A: optimizer.** SGD, Adam, AdamW, RMSProp on YOLOv8n (Adaptive optimizers get `lr0=0.001`, SGD gets `0.01`: the usual stable defaults for each.)
* **Experiment B: model capacity.** YOLOv8s trained with the winning optimizer, compared against YOLOv8n.

Runs are **resumable**: if a run already exists on Drive it is skipped, so a Colab disconnect only costs the run in progress.
Training uses transfer learning from COCO-pretrained weights, with mosaic augmentation, cosine-free default LR schedule, and AMP (mixed precision) for speed.

In [ ]:
RESULTS = {}

def load_csv(d):
    df = pd.read_csv(Path(d) / "results.csv"); df.columns = [c.strip() for c in df.columns]; return df

def summarize(name, d, meta):
    df = load_csv(d)
    fit = 0.1 * df["metrics/mAP50(B)"] + 0.9 * df["metrics/mAP50-95(B)"]
    i = int(fit.values.argmax()); r = df.iloc[i]
    best = Path(d) / "weights" / "best.pt"
    params = sum(p.numel() for p in YOLO(str(best)).model.parameters()) / 1e6
    return dict(name=name, dir=str(d), **meta, epochs_run=len(df), best_epoch=int(r["epoch"]),
                precision=float(r["metrics/precision(B)"]), recall=float(r["metrics/recall(B)"]),
                map50=float(r["metrics/mAP50(B)"]), map5095=float(r["metrics/mAP50-95(B)"]),
                params_m=round(params, 2), size_mb=round(best.stat().st_size / 1e6, 1))

def run_exp(name, weights, optimizer, lr0):
    d = RUNS / name
    info_f = d / "run_info.json"
    if (d / "weights" / "best.pt").exists() and info_f.exists():
        print(f"[skip] {name} already trained"); RESULTS[name] = json.loads(info_f.read_text()); return RESULTS[name]
    print(f"\n===== TRAINING {name}  ({weights}, {optimizer}, lr0={lr0}) =====")
    t0 = time.time()
    extra = dict(fraction=FRACTION) if FRACTION < 1 else {}
    m = YOLO(weights)
    m.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, optimizer=optimizer, lr0=lr0,
            seed=SEED, deterministic=True, project=str(RUNS), name=name, exist_ok=True,
            patience=15, cache=False, workers=0, amp=False, plots=True, verbose=False, device=DEVICE, **extra)
    d = Path(m.trainer.save_dir)
    meta = dict(weights=weights, optimizer=optimizer, lr0=lr0, train_min=round((time.time() - t0) / 60, 1))
    RESULTS[name] = summarize(name, d, meta)
    (d / "run_info.json").write_text(json.dumps(RESULTS[name], indent=2))
    print(RESULTS[name]); return RESULTS[name]

# ---------- Experiment A: optimizers on YOLOv8n
OPT_GRID = [("SGD", 0.01), ("AdamW", 0.001)]
for opt, lr in OPT_GRID:
    run_exp(f"v8n_{opt}", "yolov8n.pt", opt, lr)

In [ ]:
# ---------- Experiment B: capacity (YOLOv8s with the winning optimizer)
opt_runs = [r for k, r in RESULTS.items() if k.startswith("v8n_")]
win = max(opt_runs, key=lambda r: r["map5095"])
print(f"Winning optimizer on YOLOv8n: {win['optimizer']} (mAP50-95 = {win['map5095']:.3f})")
if RUN_V8S: run_exp(f"v8s_{win['optimizer']}", "yolov8s.pt", win["optimizer"], win["lr0"])

## 4. Experiment results (validation split)

In [ ]:
tbl = pd.DataFrame(RESULTS.values())[["name", "weights", "optimizer", "lr0", "epochs_run", "best_epoch",
                                      "precision", "recall", "map50", "map5095", "params_m", "size_mb", "train_min"]]
tbl = tbl.sort_values("map5095", ascending=False).reset_index(drop=True)
tbl.to_csv(FIGS / "experiment_table.csv", index=False)
display(tbl.round(3).style.background_gradient(subset=["map50", "map5095"], cmap="Greens").format(precision=3))

fig, axs = plt.subplots(1, 2, figsize=(18, 6))
t = tbl.sort_values("map5095")
axs[0].barh(t.name, t.map5095, color=sns.color_palette("deep", len(t)))
for y, (v, m) in enumerate(zip(t.map5095, t.train_min)): axs[0].text(v + 0.003, y, f"{v:.3f}  ({m:.0f} min)", va="center", fontsize=12)
axs[0].set_xlim(0, t.map5095.max() * 1.25); axs[0].set_title("Validation mAP50-95 (training time)")
sns.scatterplot(data=tbl, x="params_m", y="map5095", hue="name", s=300, ax=axs[1], legend=False)
for _, r in tbl.iterrows(): axs[1].annotate(r["name"], (r.params_m, r.map5095), textcoords="offset points", xytext=(8, 6), fontsize=11)
axs[1].set_xlabel("Parameters (M)"); axs[1].set_ylabel("mAP50-95"); axs[1].set_title("Accuracy vs model size")
plt.tight_layout(); save(fig, "04_experiment_summary")

In [ ]:
# --- learning curves for every run
fig, axs = plt.subplots(1, 3, figsize=(22, 6))
for name, r in RESULTS.items():
    df = load_csv(r["dir"]); x = df["epoch"]
    axs[0].plot(x, df["train/box_loss"], label=name)
    axs[1].plot(x, df["val/box_loss"], label=name)
    axs[2].plot(x, df["metrics/mAP50-95(B)"], label=name)
for a, t in zip(axs, ["Train box loss", "Validation box loss", "Validation mAP50-95"]): a.set_title(t); a.set_xlabel("epoch")
axs[2].legend(fontsize=11); plt.tight_layout(); save(fig, "05_learning_curves")

**How to read this (for the slides).** Compare the *speed of convergence* (how fast mAP rises), the *final value*, and the *gap between train and val loss*: a widening gap means overfitting. Adaptive optimizers (Adam/AdamW/RMSProp) usually converge faster early on; SGD often catches up or wins with longer schedules. State what *your* numbers show, not what the textbook says.

## 5. Final model: held-out evaluation
We pick the model with the best **validation** mAP50-95 and report its performance on the **test** split, which was never used for model selection (this avoids optimistic bias).

In [ ]:
final = max(RESULTS.values(), key=lambda r: r["map5095"])
BEST_PT = Path(final["dir"]) / "weights" / "best.pt"
print("FINAL MODEL:", final["name"], "->", BEST_PT)
model = YOLO(str(BEST_PT))
EVAL_SPLIT = "test" if "test" in SPLITS else "val"
em = model.val(data=DATA_YAML, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH, conf=0.001, iou=0.6, plots=True,
               project=str(WORK / "eval"), name="final", exist_ok=True, verbose=False, device=DEVICE)
print(f"\n[{EVAL_SPLIT}]  mAP50-95={em.box.map:.3f}  mAP50={em.box.map50:.3f}  P={em.box.mp:.3f}  R={em.box.mr:.3f}")

pc = []
for i, c in enumerate(em.box.ap_class_index):
    p, r, ap50, ap = em.box.class_result(i)
    pc.append((NAMES[int(c)], p, r, ap50, ap))
per_class = pd.DataFrame(pc, columns=["class", "precision", "recall", "AP50", "AP50-95"]).sort_values("AP50-95", ascending=False)
per_class.to_csv(FIGS / "per_class_metrics.csv", index=False)
display(per_class.round(3).style.background_gradient(subset=["AP50-95"], cmap="RdYlGn").format(precision=3))

fig, ax = plt.subplots(figsize=(11, 6.5))
sns.barplot(data=per_class, y="class", x="AP50-95", palette="viridis", ax=ax)
for i, v in enumerate(per_class["AP50-95"]): ax.text(v + 0.005, i, f"{v:.2f}", va="center")
ax.set_title(f"Per-class AP50-95 ({EVAL_SPLIT} split)"); ax.set_ylabel("")
save(fig, "06_per_class_ap")

In [ ]:
# --- Ultralytics' own diagnostic plots (confusion matrix, PR, F1) shown in the notebook
ev = Path(em.save_dir)
shown = [("confusion_matrix_normalized.png", "Normalised confusion matrix"), ("PR_curve.png", "Precision-Recall curves"), ("F1_curve.png", "F1 vs confidence")]
fig, axs = plt.subplots(1, 3, figsize=(27, 8))
for a, (f, t) in zip(axs, shown):
    if (ev / f).exists(): a.imshow(plt.imread(ev / f))
    a.set_title(t); a.axis("off")
plt.tight_layout(); save(fig, "07_diagnostics")
for f, _ in shown:
    if (ev / f).exists(): shutil.copy(ev / f, FIGS / f"eval_{f}")

**Reading the confusion matrix.** Diagonal = correct. Big off-diagonal values in the *background* row/column mean missed objects (FN) or false alarms (FP). Look for pairs like `helmet` vs `no_helmet`: those are the costly confusions in a safety setting.

## 6. Operating-point tuning
A detector outputs a confidence score per box. The threshold is a *business decision*, not a constant. We take the confidence that maximises mean F1 as the default operating point.

In [ ]:
BEST_CONF = 0.25
try:
    f1 = np.asarray(em.box.f1_curve); px = np.asarray(em.box.px); mf1 = f1.mean(0)
    BEST_CONF = float(np.clip(px[mf1.argmax()], 0.05, 0.7))
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(px, f1.T, color="lightgrey", lw=1); ax.plot(px, mf1, color="C3", lw=3, label="mean F1")
    ax.axvline(BEST_CONF, ls="--", color="k"); ax.text(BEST_CONF + 0.01, 0.05, f"best conf = {BEST_CONF:.2f}")
    ax.set_xlabel("confidence threshold"); ax.set_ylabel("F1"); ax.set_title("Choosing the confidence threshold"); ax.legend()
    save(fig, "08_f1_threshold")
except Exception as e:
    print("F1 curve unavailable, using 0.25:", e)
print("BEST_CONF =", round(BEST_CONF, 3))

In [ ]:
# --- qualitative predictions on unseen test images
test_imgs = list_images(SPLITS[EVAL_SPLIT])
sel = random.Random(7).sample(test_imgs, min(8, len(test_imgs)))
res = model.predict([str(p) for p in sel], conf=BEST_CONF, imgsz=IMGSZ, verbose=False, device=DEVICE)
fig, axs = plt.subplots(2, 4, figsize=(22, 11))
for a, r in zip(axs.ravel(), res): a.imshow(cv2.cvtColor(r.plot(line_width=2), cv2.COLOR_BGR2RGB)); a.axis("off")
plt.tight_layout(); save(fig, "09_predictions")

## 7. Application insight: from detections to compliance alerts
Managers do not care about mAP; they care about *"Which images/frames contain a safety violation, and can I trust the alert?"*

**Rule:** an image is a **VIOLATION** if the detector finds any "missing PPE" class (any class whose name starts with `no_`). We evaluate this rule against ground truth, **at image level**, and study the alert threshold trade-off: too sensitive → alarm fatigue; too lenient → missed violations.

In [ ]:
VIOLATION_IDS = [i for i, n in NAMES.items() if n.lower().startswith("no_")]
VIOLATION_NAMES = [NAMES[i] for i in VIOLATION_IDS]
print("Violation classes:", VIOLATION_NAMES)
assert VIOLATION_IDS, "No 'no_*' classes in this dataset: set VIOLATION_IDS manually."

# ground truth (image level)
g = ann[ann.split == EVAL_SPLIT]
gt_viol = g.groupby("image").cls.apply(lambda s: s.isin(VIOLATION_IDS).any())
# predictions at a LOW threshold; keep max score per violation class so we can sweep the alert threshold cheaply
paths = [str(p) for p in test_imgs]
score = {}; pred_cls_counts = {n: 0 for n in VIOLATION_NAMES}
for r in model.predict(paths, conf=0.05, imgsz=IMGSZ, batch=BATCH, stream=True, verbose=False, device=DEVICE, half=torch.cuda.is_available()):
    cls = r.boxes.cls.int().cpu().numpy(); cf = r.boxes.conf.cpu().numpy()
    m = np.isin(cls, VIOLATION_IDS)
    score[Path(r.path).name] = float(cf[m].max()) if m.any() else 0.0
sc = pd.Series(score).reindex(gt_viol.index).fillna(0.0)
y = gt_viol.values.astype(int)
print(f"{len(y)} test images, {y.mean():.1%} contain at least one real violation")

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, precision_recall_curve
ths = np.linspace(0.05, 0.9, 86)
rows = []
for t in ths:
    p, r, f, _ = precision_recall_fscore_support(y, (sc.values >= t).astype(int), average="binary", zero_division=0)
    rows.append((t, p, r, f))
sweep = pd.DataFrame(rows, columns=["thr", "precision", "recall", "f1"])
t_f1 = float(sweep.loc[sweep.f1.idxmax(), "thr"])
cand = sweep[sweep.recall >= 0.95]
t_safe = float(cand.thr.max()) if len(cand) else float(sweep.loc[sweep.recall.idxmax(), "thr"])

fig, axs = plt.subplots(1, 2, figsize=(19, 6.5))
axs[0].plot(sweep.thr, sweep.precision, label="precision"); axs[0].plot(sweep.thr, sweep.recall, label="recall"); axs[0].plot(sweep.thr, sweep.f1, label="F1", lw=3)
axs[0].axvline(t_f1, ls="--", c="k"); axs[0].text(t_f1 + .01, .05, f"balanced\n{t_f1:.2f}")
axs[0].axvline(t_safe, ls=":", c="r"); axs[0].text(t_safe + .01, .25, f"safety-first\n(recall>=95%)\n{t_safe:.2f}", color="r")
axs[0].set_xlabel("alert threshold"); axs[0].set_title("Image-level violation alert trade-off"); axs[0].legend()
cm = confusion_matrix(y, (sc.values >= t_f1).astype(int))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axs[1], xticklabels=["OK", "ALERT"], yticklabels=["Compliant", "Violation"], annot_kws={"size": 22})
axs[1].set_title(f"Alert confusion matrix @ {t_f1:.2f}"); axs[1].set_xlabel("system says"); axs[1].set_ylabel("reality")
plt.tight_layout(); save(fig, "10_violation_alerts")

def at(t):
    p, r, f, _ = precision_recall_fscore_support(y, (sc.values >= t).astype(int), average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, (sc.values >= t).astype(int), labels=[0, 1]).ravel()
    return dict(threshold=round(t, 2), precision=round(p, 3), recall=round(r, 3), f1=round(f, 3), caught=tp, missed=fn, false_alarms=fp)
op = pd.DataFrame([dict(mode="balanced (max F1)", **at(t_f1)), dict(mode="safety-first (recall>=95%)", **at(t_safe))])
op.to_csv(FIGS / "alert_operating_points.csv", index=False); display(op)

In [ ]:
# --- which violation types does the system find / miss?
gt_type = {n: g[g.cls == i].image.nunique() for i, n in zip(VIOLATION_IDS, VIOLATION_NAMES)}
pc_type = {n: 0 for n in VIOLATION_NAMES}
for r in model.predict(paths, conf=BEST_CONF, imgsz=IMGSZ, batch=BATCH, stream=True, verbose=False, device=DEVICE, half=torch.cuda.is_available()):
    present = set(r.boxes.cls.int().cpu().tolist())
    for i, n in zip(VIOLATION_IDS, VIOLATION_NAMES):
        if i in present: pc_type[n] += 1
vt = pd.DataFrame({"actual (images)": gt_type, "detected (images)": pc_type})
fig, ax = plt.subplots(figsize=(11, 6)); vt.plot.bar(ax=ax, rot=0)
ax.set_title("Violation types: ground truth vs system"); ax.set_ylabel("# test images")
save(fig, "11_violation_types"); display(vt)

## 8. Deployment optimisation
Accuracy is only useful if it runs fast enough on real hardware. We measure **end-to-end latency** (pre-processing + inference + NMS) for:
* PyTorch FP32 on GPU, then **FP16 (half precision)** on GPU
* PyTorch on CPU vs **ONNX** export on CPU (what an edge box without a GPU would run)

In [ ]:
bench_imgs = [cv2.imread(str(p)) for p in random.Random(3).sample(test_imgs, min(40, len(test_imgs)))]

def bench(m, device, half=False, n=None):
    imgs = bench_imgs[:n] if n else bench_imgs
    for _ in range(3): m.predict(imgs[0], imgsz=IMGSZ, device=device, half=half, verbose=False)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    for im in imgs: m.predict(im, imgsz=IMGSZ, device=device, half=half, verbose=False, conf=BEST_CONF)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / len(imgs) * 1000
    return round(ms, 1), round(1000 / ms, 1)

rows = []
if torch.cuda.is_available():
    rows.append(("PyTorch FP32 - GPU", *bench(model, 0, False)))
    rows.append(("PyTorch FP16 - GPU", *bench(model, 0, True)))
rows.append(("PyTorch FP32 - CPU", *bench(model, "cpu", False, n=15)))
try:
    onnx_path = model.export(format="onnx", imgsz=IMGSZ, simplify=True, verbose=False)
    onnx_model = YOLO(onnx_path, task="detect")
    rows.append(("ONNX - CPU", *bench(onnx_model, "cpu", False, n=15)))
    print("ONNX size: %.1f MB vs PT %.1f MB" % (Path(onnx_path).stat().st_size / 1e6, BEST_PT.stat().st_size / 1e6))
except Exception as e:
    print("ONNX export skipped:", e)
lat = pd.DataFrame(rows, columns=["setup", "ms_per_image", "FPS"])
lat.to_csv(FIGS / "latency_table.csv", index=False); display(lat)

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=lat, y="setup", x="FPS", palette="crest", ax=ax)
for i, (f, m) in enumerate(zip(lat.FPS, lat.ms_per_image)): ax.text(f + 0.5, i, f"{f:.0f} FPS ({m:.0f} ms)", va="center")
ax.set_xlim(0, lat.FPS.max() * 1.25); ax.set_ylabel(""); ax.set_title("Inference speed (end-to-end, batch=1)")
save(fig, "12_latency")

## 9. Demo video for the presentation
Annotates frames with detections and a live **COMPLIANT / VIOLATION** banner. To use your own footage (recommended: 10–20 s of workers wearing PPE, e.g. a stock clip), upload it and set `VIDEO_PATH`. Otherwise the test images are stitched into a clip.

In [ ]:
VIDEO_PATH = None     # e.g. "/content/site_clip.mp4"
OUT_RAW, OUT_MP4 = str(WORK / "demo_raw.mp4"), str(WORK / "compliance_demo.mp4")

def letterbox(im, size=(960, 540)):
    h, w = im.shape[:2]; s = min(size[0] / w, size[1] / h)
    im = cv2.resize(im, (int(w * s), int(h * s))); canvas = np.full((size[1], size[0], 3), 20, np.uint8)
    y0, x0 = (size[1] - im.shape[0]) // 2, (size[0] - im.shape[1]) // 2
    canvas[y0:y0 + im.shape[0], x0:x0 + im.shape[1]] = im; return canvas

def frames():
    if VIDEO_PATH:
        cap = cv2.VideoCapture(VIDEO_PATH)
        while True:
            ok, f = cap.read()
            if not ok: break
            yield letterbox(f)
    else:
        for p in test_imgs[:40]:
            f = letterbox(cv2.imread(str(p)))
            for _ in range(4): yield f      # hold each image ~0.5 s at 15 fps

vw = cv2.VideoWriter(OUT_RAW, cv2.VideoWriter_fourcc(*"mp4v"), 15, (960, 540)); n = 0; t0 = time.perf_counter(); nviol = 0
for f in frames():
    r = model.predict(f, conf=BEST_CONF, imgsz=IMGSZ, verbose=False, device=DEVICE, half=torch.cuda.is_available())[0]
    out = r.plot(line_width=2)
    cls = r.boxes.cls.int().cpu().tolist(); viol = [NAMES[c] for c in cls if c in VIOLATION_IDS]
    bad = len(viol) > 0; nviol += bad
    cv2.rectangle(out, (0, 0), (960, 46), (0, 0, 190) if bad else (30, 140, 40), -1)
    cv2.putText(out, ("VIOLATION: " + ", ".join(sorted(set(viol)))) if bad else "COMPLIANT", (14, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.95, (255, 255, 255), 2)
    vw.write(out); n += 1
vw.release()
print(f"{n} frames, {n / (time.perf_counter() - t0):.1f} FPS incl. drawing, {nviol / max(n,1):.0%} of frames flagged")
try:
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUT_RAW, "-vcodec", "libx264", "-pix_fmt", "yuv420p", OUT_MP4], check=True)
except Exception as e:
    print("ffmpeg convert failed, using raw mp4:", e); OUT_MP4 = OUT_RAW
print("Saved:", OUT_MP4)
try:
    from google.colab import files; files.download(OUT_MP4)
except Exception: pass

## 10. Export everything for the slides

In [ ]:
shutil.copy(BEST_PT, WORK / "best_model.pt")
summary_txt = f"""FINAL MODEL : {final['name']}  ({final['optimizer']}, lr0={final['lr0']}, {final['params_m']} M params, {final['size_mb']} MB)
{EVAL_SPLIT.upper()} mAP50-95 : {em.box.map:.3f}   mAP50: {em.box.map50:.3f}   P: {em.box.mp:.3f}   R: {em.box.mr:.3f}
Best conf threshold (F1)   : {BEST_CONF:.2f}
Alert (balanced)           : {op.iloc[0].to_dict()}
Alert (safety-first)       : {op.iloc[1].to_dict()}
Speed                      : {lat.to_dict('records')}
"""
(FIGS / "SUMMARY.txt").write_text(summary_txt); print(summary_txt)
zip_path = shutil.make_archive(str(WORK / "yolo_deliverables"), "zip", FIGS)
print("Figures + tables zipped ->", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception: pass

### Conclusions (fill in from *your* numbers before presenting)
1. **Optimizer:** which won, and by how much? Did it converge faster or just end higher?
2. **Capacity:** was YOLOv8s worth the extra parameters/latency over YOLOv8n?
3. **Weakest classes:** which ones, and is it because of size, rarity or visual similarity (see section 2 and the confusion matrix)?
4. **Business value:** at the safety-first threshold, what share of violations are caught and how many false alarms per N images?
5. **Deployment:** does the model reach real-time (≥ 25 FPS) on GPU? What does ONNX-CPU give for an edge device?
6. **Limitations / next steps:** small dataset, one camera style, no tracking across frames, privacy considerations (faces), person-level (not image-level) alerts, a tracker (e.g. ByteTrack) to count unique violators.